# Topic 06 — Where is the saturation knee?

The Topic 05 service answers one ticket fine. This lab asks what happens when many
clients arrive at once: run controlled load waves at 8/32/64/128 concurrent clients,
keep a per-request journal where errors are data, build error-aware plots, locate the
knee, and defend one operating point under an SLO.

You will implement seven typed functions: `kv_budget_bytes`, `audit_records`,
`throughput_by_wave`, `ttft_percentiles`, `detect_knee`, `choose_operating_point`
and `interference_delta`. Everything else is supplied infrastructure.

Two profiles, one notebook:

- **mock** (default, CPU, minutes): a declared synthetic queueing model stands in for
  the server. It validates plumbing, journals, analysis and plots. It is NOT a
  measurement of any real system.
- **gpu** (instructor/owner run only): a pinned vLLM service answers the same waves
  with the frozen public corpus. Only that profile produces measured evidence.

Workload contract: 2 contact hours + 3–5 own hours; ≤ 90 minutes of GPU time when
the owner runs the gpu profile on a T4. Raw journals are preserved; never edit them.


In [ ]:
import json, os, time, threading, hashlib, urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import TypedDict, Sequence

PROFILE = os.environ.get('LSM_T06_PROFILE', 'mock')  # 'mock' | 'gpu' (owner-run only)
ATTEMPT = 'attempt-001'
RESULTS = os.path.join('results', PROFILE, ATTEMPT)
os.makedirs(RESULTS, exist_ok=True)
print(f'profile={PROFILE}  results dir={RESULTS}')

In [ ]:
class ModelConfig(TypedDict):
    """Serving-relevant model facts, declared once and shared by every calculation."""
    n_layers: int; n_kv_heads: int; head_dim: int; dtype_bytes: int

MODEL_FACTS: ModelConfig = dict(n_layers=36, n_kv_heads=2, head_dim=128, dtype_bytes=2)

class RequestRecord(TypedDict):
    """One client-observed request. Timestamps are client-side monotonic seconds."""
    request_id: str; wave: int; client_id: int; app_id: str; case_id: str
    input_tokens: int; max_tokens: int
    arrival_ts: float; first_byte_ts: float | None; terminal_ts: float
    output_tokens: int; http_status: int | None; outcome: str

class Percentiles(TypedDict):
    """Nearest-rank TTFT percentiles for one wave, completed requests only."""
    wave: int; count: int; p50_ms: float; p95_ms: float

class AuditSummary(TypedDict):
    """Damage classification of one wave's journal; nothing is repaired."""
    records: int; unique_ids: int; missing_terminal: int; duplicate_ids: int
    foreign_waves: int; impossible_time: int; complete: int

class Decision(TypedDict):
    """The chosen operating wave with its evidence and named binding constraint."""
    wave: int; throughput_tok_per_s: float; ttft_p95_ms: float; error_rate: float
    satisfies_slo: bool; reason: str

class InterferenceReport(TypedDict):
    """A-app degradation when B shares the server, plus B's completed count."""
    a_throughput_ratio: float; a_ttft_p95_ratio: float
    a_completed_alone: int; a_completed_shared: int; b_completed_shared: int

def save_jsonl(path: str, rows: Sequence[dict]) -> str:
    """Write a JSONL journal and return its sha256 hex digest.

    Parameters
    ----------
    path : str
        Destination file path, overwritten.
    rows : sequence of dict
        Records to serialize, one JSON object per line.

    Returns
    -------
    str
        sha256 of the written bytes, for provenance receipts.
    """
    with open(path, 'w', encoding='utf-8') as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + chr(10))
    return hashlib.sha256(Path_bytes(path)).hexdigest()

def Path_bytes(path: str) -> bytes:
    """Read a file as raw bytes.

    Parameters
    ----------
    path : str
        File to read.

    Returns
    -------
    bytes
        Exact file content, for hashing.
    """
    with open(path, 'rb') as fh:
        return fh.read()

def load_jsonl(path: str) -> list[dict]:
    """Load a JSONL file into a list of dicts.

    Parameters
    ----------
    path : str
        Journal file, one JSON object per line.

    Returns
    -------
    list of dict
        Parsed records, in file order.
    """
    with open(path, encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

## Block 0 — Predict before you measure

Worked arithmetic for the pinned model (Qwen2.5-3B-Instruct, FP16, 36 layers, 2 KV
heads, head width 128): one retained position costs
2 (K and V) × 2 heads × 128 width × 2 bytes × 36 layers = **36 864 bytes ≈ 36 KiB**.
That is arithmetic on declared facts — not measured GPU memory.

A T4 exposes ~15 360 MiB. Weights of a 3B model in FP16 occupy ~6.2 GiB (declared
assumption). Before any load run, compute how many simultaneous 2 048-token contexts
fit the remainder — then freeze that prediction. You will compare it with the
measured knee in Block 2 and explain the gap.

In [ ]:
def kv_budget_bytes(cfg: ModelConfig, context_tokens: int) -> int:
    """Total KV-cache bytes one request occupies for a fully retained context.

    Parameters
    ----------
    cfg : ModelConfig
        Declared model facts: layers, KV heads, head width, bytes per element.
    context_tokens : int
        Retained positions for one request, >= 0.

    Returns
    -------
    int
        Bytes for K and V together across all layers for this one request.

    Raises
    ------
    ValueError
        If ``context_tokens`` is negative or any cfg field is not a positive int.

    Notes
    -----
    Arithmetic on declared facts. It is an upper-bound ceiling, not vLLM's
    allocation policy: fragmentation, activations and swap are not included.
    """
    raise NotImplementedError('TODO: derive the per-position and per-request byte cost')

In [ ]:
assert kv_budget_bytes(MODEL_FACTS, 1) == 36_864
assert kv_budget_bytes(MODEL_FACTS, 0) == 0
try:
    kv_budget_bytes(MODEL_FACTS, -3); raise SystemExit('negative context must raise')
except ValueError:
    pass
print('kv_budget_bytes: per-position and edge cases match the worked arithmetic')

In [ ]:
T4_USABLE_BYTES = 15_360 * 1024 * 1024
WEIGHT_BYTES = int(6.2 * 1024**3)
CTX_TOKENS = 2048

per_request = kv_budget_bytes(MODEL_FACTS, CTX_TOKENS)
ceiling = (T4_USABLE_BYTES - WEIGHT_BYTES) // per_request
predicted_knee = 32   # TODO: your prediction: the largest wave you expect to stay healthy
prediction_rationale = 'TODO: one paragraph — which resource do you expect to bind first, and why'
pred = dict(per_request_bytes=per_request, ceiling_requests=ceiling,
            predicted_knee_wave=predicted_knee, rationale=prediction_rationale,
            profile=PROFILE, frozen_at=time.time())
with open(os.path.join(RESULTS, 'predictions.json'), 'w') as fh:
    json.dump(pred, fh, indent=1)
print(f'ceiling (arithmetic) = {ceiling} requests at {CTX_TOKENS} tokens; prediction frozen')

## Block 1 — Measure: waves, journal, outcomes

A **wave** runs N concurrent closed-loop clients; each client sends its next request
only after the previous one terminates (no retries, ever — a retry would silently
rewrite demand). Every request writes one journal record with client-side monotonic
timestamps and a terminal outcome:

`completed` (HTTP 200 + full non-empty output) · `timeout` · `server_error` (5xx) ·
`connection_error` · `truncated` (finish reason = length).

Errors are data: they are counted per wave and plotted, never dropped from the
denominator. The supplied driver classifies outcomes; you audit and analyse.

In [ ]:
class MockQueueServer:
    """Declared synthetic contention model. Validation plumbing, NOT a measurement.

    Per request: admission queue delay 0.9 * in_flight**2 ms, prefill 1.2 ms/token,
    decode 8 ms/token * (1 + 0.06 * (in_flight - 1)). Deterministic given the
    in-flight count observed at admission.
    """
    def __init__(self) -> None:
        """Create a server with zero requests in flight.

        Returns
        -------
        None
            Stateful helper; see request().
        """
        self._lock = threading.Lock()
        self._in_flight = 0

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Serve one synthetic request.

        Parameters
        ----------
        case : dict
            Load case with input_tokens.
        max_tokens : int
            Requested output tokens.

        Returns
        -------
        tuple of (str, int)
            Output text and output token count.
        """
        input_tokens = case['input_tokens']
        with self._lock:
            n = self._in_flight = self._in_flight + 1
        try:
            queue_ms = 0.9 * n * n
            ttft = (0.15 + queue_ms / 1000 + 1.2 * input_tokens / 1000)
            per_tok = 0.008 * (1 + 0.06 * (n - 1))
            time.sleep(ttft)
            time.sleep(per_tok * max_tokens)
            return 'SYNTHETIC-MOCK-OUTPUT', max_tokens
        finally:
            with self._lock:
                self._in_flight -= 1

def run_wave(runner: object, app_id: str, cases: Sequence[dict], wave: int, per_client: int,
             timeout_s: float) -> list[RequestRecord]:
    """Run one closed-loop wave of `wave` clients, `per_client` requests each.

    Parameters
    ----------
    runner : object
        Service with .request(case, max_tokens); mock here, HTTP on gpu.
    app_id : str
        Application label written into every record.
    cases : sequence of dict
        Load cases with case_id and input_tokens (mock) or rendered (gpu).
    wave : int
        Concurrency: number of simultaneous closed-loop clients.
    per_client : int
        Sequential requests each client sends.
    timeout_s : float
        Per-request wall budget before the client gives up.

    Returns
    -------
    list of RequestRecord
        One record per terminated attempt, outcomes classified.
    """
    records: list[RequestRecord] = []
    lock = threading.Lock()

    def client(cid: int) -> None:
        """Send this client's sequential requests and journal each attempt.

        Parameters
        ----------
        cid : int
            Client index inside the wave.

        Returns
        -------
        None
            Results are appended to the shared records list.
        """
        for seq in range(per_client):
            case = cases[(cid * per_client + seq) % len(cases)]
            rid = f'w{wave}-c{cid}-r{seq}'
            t0 = time.monotonic()
            outcome, status, out_tok, fb = 'connection_error', None, 0, None
            try:
                text, out_tok = runner.request(case, 32)
                fb = time.monotonic()
                outcome, status = ('completed', 200) if out_tok > 0 else ('truncated', 200)
                if time.monotonic() - t0 > timeout_s:
                    outcome = 'timeout'
            except TimeoutError:
                outcome = 'timeout'
            except Exception as exc:  # noqa: BLE001 - transport failures are data
                outcome = 'server_error' if '5xx' in str(exc) else 'connection_error'
            rec = RequestRecord(request_id=rid, wave=wave, client_id=cid, app_id=app_id,
                                case_id=case['case_id'], input_tokens=case['input_tokens'],
                                max_tokens=32, arrival_ts=t0, first_byte_ts=fb,
                                terminal_ts=time.monotonic(), output_tokens=out_tok,
                                http_status=status, outcome=outcome)
            if time.monotonic() - t0 > timeout_s and outcome == 'completed':
                rec['outcome'] = 'timeout'
            with lock:
                records.append(rec)

    with ThreadPoolExecutor(max_workers=wave) as pool:
        list(pool.map(client, range(wave)))
    return records

def run_shared_wave(server: object, b_cap_k: int, a_cases: Sequence[dict], b_cases: Sequence[dict],
                    n_a: int, n_b: int, per_client: int, timeout_s: float) -> list[RequestRecord]:
    """One concurrent wave: n_a A-clients plus n_b B-clients capped at b_cap_k.

    Parameters
    ----------
    server : object
        Runner shared by both applications; overlap IS the interference.
    b_cap_k : int
        In-flight cap for B, one of 1/2/4/8/16.
    a_cases : sequence of dict
        Load cases for application A.
    b_cases : sequence of dict
        Load cases for application B.
    n_a : int
        A client count.
    n_b : int
        B client count.
    per_client : int
        Sequential requests per client.
    timeout_s : float
        Per-request wall budget.

    Returns
    -------
    list of RequestRecord
        Merged A+B journal; app_id and the B- request-id prefix distinguish them.
    """
    capped = InFlightCapRunner(server, b_cap_k)
    with ThreadPoolExecutor(max_workers=n_a + n_b) as pool:
        fut_a = pool.submit(run_wave, server, 'A', a_cases, n_a, per_client, timeout_s)
        fut_b = pool.submit(run_wave, capped, 'B', b_cases, n_b, per_client, timeout_s)
        a_recs, b_recs = fut_a.result(), fut_b.result()
    for r in b_recs:
        r['request_id'] = 'B-' + r['request_id']
    return a_recs + b_recs

In [ ]:
# GPU PROFILE ONLY. Pinned vLLM install on a fresh T4 runtime (~3-6 min).
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import subprocess
    out = subprocess.run(['pip', 'install', '-q', 'vllm==0.30.0'],
                         capture_output=True, text=True)
    print('pip exit', out.returncode, (out.stderr or '')[-300:])
    assert out.returncode == 0, 'vLLM install failed; see stderr above'


In [ ]:
# GPU PROFILE ONLY. Owned child vLLM service; teardown lives in t06-gpu-stop.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import subprocess  # noqa: E402
    SERVER_CMD = ['python', '-m', 'vllm.entrypoints.openai.api_server',
                  '--model', 'Qwen/Qwen2.5-3B-Instruct',
                  '--revision', 'aa8e72537993ba99e69dfaafa59ed015b17504d1',
                  '--served-model-name', 'submission', '--dtype', 'float16',
                  '--max-model-len', '2048', '--port', '8017']
    server = subprocess.Popen(SERVER_CMD,
                              stdout=open(os.path.join(RESULTS, 'server.log'), 'w'),
                              stderr=subprocess.STDOUT)
    print('server pid', server.pid, '- the next cell waits for /health')

In [ ]:
# GPU PROFILE ONLY. Bounded readiness wait; never wave into a dead server.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import urllib.error  # noqa: E402
    waited = 0
    healthy = False
    deadline = time.monotonic() + 900
    while not healthy:
        try:
            with urllib.request.urlopen('http://127.0.0.1:8017/health', timeout=5) as r:
                healthy = r.status == 200
        except (urllib.error.URLError, OSError):
            pass
        assert time.monotonic() < deadline, 'service did not become healthy in 900 s'
        if not healthy:
            time.sleep(5); waited += 5
    print(f'service healthy; waited about {waited} s')

In [ ]:
# GPU PROFILE ONLY. Frozen public corpus (pinned example revision), D/V splits.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    PINNED_CORPUS_URL = (
        'https://raw.githubusercontent.com/levshaazz/llm-serving-mastery-submission-example/'
        '8525bd29959562db69800bf453555155f2ade668/classroom/topic06-purpose-v4/corpus.jsonl')
    CORPUS_LOCAL = os.path.join(RESULTS, 'corpus.jsonl')
    if not os.path.exists(CORPUS_LOCAL):
        urllib.request.urlretrieve(PINNED_CORPUS_URL, CORPUS_LOCAL)
    rows = load_jsonl(CORPUS_LOCAL)
    def _cases(app: str, split: str) -> list[dict]:
        """Select load cases for one application and split.

        Parameters
        ----------
        app : str
            Application label, 'A' or 'B'.
        split : str
            Corpus split, 'D' (development) or 'V' (held-out).

        Returns
        -------
        list of dict
            Cases with case_id, input_tokens and rendered prompt.
        """
        return [dict(case_id=r['case_id'], input_tokens=r['input_tokens'],
                     rendered=r['rendered_text'])
                for r in rows if r['split'] == split and r['app_id'] == app
                and r['length_bin'] == 'short']
    gpu_cases_A, gpu_cases_B = _cases('A', 'D'), _cases('B', 'D')
    heldout_A, heldout_B = _cases('A', 'V'), _cases('B', 'V')
    print(f'dev A={len(gpu_cases_A)} B={len(gpu_cases_B)}  '
          f'heldout A={len(heldout_A)} B={len(heldout_B)}')

In [ ]:
class HttpRunner:
    """Synchronous completions client for the local pinned service (case dict in)."""
    def __init__(self, port: int = 8017) -> None:
        """Point the client at one local service port.

        Parameters
        ----------
        port : int, default 8017
            Port of the owned vLLM service.

        Returns
        -------
        None
            Connectionless helper; see request().
        """
        self.url = f'http://127.0.0.1:{port}/v1/completions'

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Complete one rendered prompt.

        Parameters
        ----------
        case : dict
            Load case with the rendered prompt under 'rendered'.
        max_tokens : int
            Output token cap.

        Returns
        -------
        tuple of (str, int)
            Output text and completion token count.
        """
        body = json.dumps(dict(model='submission', prompt=case['rendered'],
                               max_tokens=max_tokens, temperature=0.0)).encode()
        req = urllib.request.Request(self.url, data=body,
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=120) as resp:
            data = json.loads(resp.read())
        return data['choices'][0]['text'], data['usage']['completion_tokens']

In [ ]:
WAVES = [8, 32, 64, 128]
if PROFILE == 'gpu':
    RUNNER, CASES, TMO = HttpRunner(), gpu_cases_A, 60.0
else:
    MOCK_CASES = [dict(case_id=f'synth-{i}', input_tokens=n)
                  for i, n in enumerate([100, 110, 120] * 40)]
    RUNNER, CASES, TMO = MockQueueServer(), MOCK_CASES, 12.0
all_records: list[RequestRecord] = []
lines: list[str] = []
for w in WAVES:
    recs = run_wave(RUNNER, 'A', CASES, w, per_client=3, timeout_s=TMO)
    all_records.extend(recs)
    counts = {o: sum(1 for r in recs if r['outcome'] == o) for o in sorted({r['outcome'] for r in recs})}
    lines.append(f'wave {w:3d}: {len(recs)} records  outcomes={counts}')
print(chr(10).join(lines))

In [ ]:
journal_path = os.path.join(RESULTS, 'raw-journal.jsonl')
digest = save_jsonl(journal_path, all_records)
print(f'{len(all_records)} records -> {journal_path}  sha256={digest[:16]}…')

## Block 2 — Audit, then analyse

Journals lie in four documented ways: **missing** terminal records, **duplicate**
request IDs, **foreign** wave tags and **impossible** timestamps (terminal before
arrival). An unaudited journal invalidates every number built on it. Audit first,
then compute per-wave throughput, TTFT percentiles and the knee.

In [ ]:
def audit_records(records: Sequence[RequestRecord], wave: int | None = None) -> AuditSummary:
    """Audit one wave's journal (or all waves when ``wave`` is None).

    Parameters
    ----------
    records : sequence of RequestRecord
        Raw journal records, possibly damaged.
    wave : int or None
        Restrict to this wave label; None audits everything.

    Returns
    -------
    AuditSummary
        records (considered), unique_ids, missing_terminal (first_byte or
        terminal absent), duplicate_ids, foreign_waves (record['wave'] != wave
        when restricted), impossible_time (terminal < arrival), complete.

    Notes
    -----
    A record counts as complete only when it is unique, in-wave, fully timed and
    its timestamps are ordered. Nothing is ever silently repaired or dropped.
    """
    raise NotImplementedError('TODO: classify each record; never repair it')

In [ ]:
def rec(rid: str, wave: int = 8, arr: float = 1.0, fb: float = 1.1, term: float = 1.2) -> RequestRecord:
    """Build one synthetic journal record for the audit fixture.

    Parameters
    ----------
    rid : str
        Request identity.
    wave : int, default 8
        Wave label.
    arr : float, default 1.0
        Arrival timestamp, monotonic seconds.
    fb : float or None, default 1.1
        First-byte timestamp; None simulates a missing field.
    term : float, default 1.2
        Terminal timestamp.

    Returns
    -------
    RequestRecord
        A completed-style record with the given clocks.
    """
    return RequestRecord(request_id=rid, wave=wave, client_id=0, app_id='A', case_id='c',
                         input_tokens=100, max_tokens=32, arrival_ts=arr, first_byte_ts=fb,
                         terminal_ts=term, output_tokens=32, http_status=200, outcome='completed')

fixture = [rec('ok-1'), rec('ok-2'),
           rec('dup', ), rec('dup'),
           rec('no-fb', fb=None),
           rec('back', term=0.5),
           rec('foreign', wave=999)]
got = audit_records(fixture, wave=8)
expected = AuditSummary(records=6, unique_ids=5, missing_terminal=1, duplicate_ids=1,
                        foreign_waves=1, impossible_time=1, complete=3)
assert got == expected, (got, expected)
print('audit_records: damaged-journal fixture classified exactly')

In [ ]:
print(chr(10).join(f'{w} {audit_records(all_records, wave=w)}' for w in WAVES))

In [ ]:
def throughput_by_wave(records: Sequence[RequestRecord]) -> dict[int, float]:
    """Output throughput per wave: completed output tokens / wave wall seconds.

    Parameters
    ----------
    records : sequence of RequestRecord
        Audited journal records across waves.

    Returns
    -------
    dict[int, float]
        wave -> tokens per second. The wall window runs from the earliest
        arrival to the latest terminal IN that wave; only records with outcome
        'completed' contribute tokens; failed requests still consume wall time.

    Raises
    ------
    ValueError
        If a wave contains fewer than two records or no completed record.
    """
    raise NotImplementedError('TODO: tokens of completed requests over the wave wall window')

In [ ]:
two_waves = [rec('a', wave=8, arr=0.0, fb=0.1, term=1.0),
             rec('b', wave=8, arr=0.2, fb=0.3, term=2.0),   # wall=2.0s
             rec('c', wave=32, arr=0.0, fb=0.1, term=4.0),
             rec('d', wave=32, arr=0.0, fb=0.1, term=1.0)]  # wall=4.0s
got_tp = throughput_by_wave(two_waves)
assert got_tp[8] == 64 / 2.0 and got_tp[32] == 64 / 4.0, got_tp
print('throughput_by_wave: hand-computed windows match')

In [ ]:
def ttft_percentiles(records: Sequence[RequestRecord], wave: int,
                   percentiles: tuple[int, ...] = (50, 95)) -> Percentiles:
    """Client-observed TTFT percentiles for one wave, completed requests only.

    Parameters
    ----------
    records : sequence of RequestRecord
        Audited journal.
    wave : int
        Wave label to restrict to.
    percentiles : tuple[int, ...], default (50, 95)
        Nearest-rank percentiles to report.

    Returns
    -------
    Percentiles
        wave, count (denominator: completed with first_byte), p50_ms, p95_ms.

    Raises
    ------
    ValueError
        If the wave has no completed request with a first-byte timestamp.
    """
    raise NotImplementedError('TODO: nearest-rank percentiles over completed TTFTs')

In [ ]:
tt = [rec(f't{i}', wave=8, arr=0.0, fb=0.010 * (i + 1), term=0.5) for i in range(100)]
got_p = ttft_percentiles(tt, 8)
assert got_p['count'] == 100 and abs(got_p['p50_ms'] - 500.0) < 1e-9 and abs(got_p['p95_ms'] - 950.0) < 1e-6, got_p
print('ttft_percentiles: nearest-rank p50/p95 match hand computation')

In [ ]:
def detect_knee(throughput: dict[int, float], min_gain: float) -> int:
    """Last wave whose throughput still gained >= min_gain over the previous wave.

    Parameters
    ----------
    throughput : dict[int, float]
        wave -> tokens/s, at least two waves, keys interpreted in sorted order.
    min_gain : float
        Required relative gain (0.5 means +50%%) to still count as growing.

    Returns
    -------
    int
        The knee wave: the largest wave whose gain over its predecessor is
        still >= min_gain. Waves before any qualifying gain return the first key.

    Raises
    ------
    ValueError
        If fewer than two waves or any value is not a positive finite number.
    """
    raise NotImplementedError('TODO: walk sorted waves and keep the last strong gain')

In [ ]:
curve = {8: 100.0, 32: 200.0, 64: 260.0, 128: 250.0}
assert detect_knee(curve, 0.5) == 32   # +100%% then +30%%: knee at 32
assert detect_knee(curve, 0.2) == 64   # with a weaker bar, 64 still gained 30%%
print('detect_knee: gain thresholds behave on the synthetic curve')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt  # noqa: E402

def plot_curve(xs: list[int], ys: list[float], ylabel: str, path: str,
               title: str = '') -> None:
    """Draw one labelled curve per figure; axes, units and population are mandatory.

    Parameters
    ----------
    xs : list of int
        Concurrency values.
    ys : list of float
        Metric values, same length as xs.
    ylabel : str
        Y-axis label with units.
    path : str
        File name inside RESULTS.
    title : str, default ''
        Optional figure title.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(xs, ys, marker='o')
    ax.set_xlabel('concurrent clients (closed loop)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

def outcome_shares(records: Sequence[RequestRecord], waves: Sequence[int], path: str) -> None:
    """Stack per-wave outcome shares; denominators include every record.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    waves : sequence of int
        Wave labels to plot.
    path : str
        File name inside RESULTS.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    kinds = ['completed', 'timeout', 'server_error', 'connection_error', 'truncated']
    fig, ax = plt.subplots(figsize=(6, 4))
    bottoms = [0.0] * len(waves)
    for kind in kinds:
        shares = []
        for w in waves:
            rs = [r for r in records if r['wave'] == w]
            shares.append(sum(1 for r in rs if r['outcome'] == kind) / len(rs) if rs else 0.0)
        ax.bar([str(w) for w in waves], shares, bottom=bottoms, label=kind)
        bottoms = [b + s for b, s in zip(bottoms, shares)]
    ax.set_xlabel('concurrent clients'); ax.set_ylabel('share of terminated requests')
    ax.legend(); fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
tp = throughput_by_wave(all_records)
plot_curve(WAVES, [tp[w] for w in WAVES], 'output tokens / s (completed)',
           'throughput.png', title='Throughput vs concurrency (mock: declared model)')

In [ ]:
p95 = [ttft_percentiles(all_records, w)['p95_ms'] for w in WAVES]
plot_curve(WAVES, p95, 'TTFT p95, ms (completed)', 'ttft-p95.png',
           title='Tail first-token latency vs concurrency (mock)')

In [ ]:
outcome_shares(all_records, WAVES, 'outcome-shares.png')

In [ ]:
tp = throughput_by_wave(all_records)
knee = detect_knee(tp, min_gain=0.5)
pred = json.load(open(os.path.join(RESULTS, 'predictions.json')))
print(f'measured knee wave = {knee}   (' +
      ', '.join(f'{w}: {tp[w]:.0f} tok/s' for w in WAVES) + ')' + chr(10) +
      "frozen prediction = " + str(pred['predicted_knee_wave']) + ';  ' +
      'arithmetic ceiling = ' + str(pred['ceiling_requests']) + ' requests')

## Block 3 — Choose an operating point under an SLO

The course judge scores the geometric mean of output throughput at 32 and 64
concurrent clients and gates p95 TTFT at 64 clients (≤ 0.8 s on the current
contract). Your lab SLO mirrors that shape. Choose the largest wave that satisfies
the SLO with a clean error profile — then defend it in writing against your Block 0
prediction: which resource bound first, and what explains the gap?

In [ ]:
def choose_operating_point(throughput: dict[int, float], ttft_p95_ms: dict[int, float],
                          error_rate: dict[int, float], ttft_slo_ms: float,
                          max_error_rate: float) -> Decision:
    """Largest wave whose p95 TTFT and error rate both satisfy the SLO.

    Parameters
    ----------
    throughput, ttft_p95_ms, error_rate : dict[int, float]
        Per-wave measured curves (all waves present in each).
    ttft_slo_ms : float
        Maximum allowed p95 TTFT in milliseconds.
    max_error_rate : float
        Maximum allowed share of non-completed records, in [0, 1].

    Returns
    -------
    Decision
        The chosen wave with its numbers and a one-sentence reason naming the
        binding constraint. When no wave qualifies, wave is -1 and the reason
        must say which constraint failed everywhere.

    Raises
    ------
    ValueError
        If the dicts disagree on wave labels or any input is out of range.
    """
    raise NotImplementedError('TODO: largest SLO-satisfying wave, binding constraint named')

In [ ]:
d = choose_operating_point(
    throughput={8: 100.0, 32: 200.0, 64: 260.0},
    ttft_p95_ms={8: 200.0, 32: 700.0, 64: 1500.0},
    error_rate={8: 0.0, 32: 0.01, 64: 0.02},
    ttft_slo_ms=800.0, max_error_rate=0.05)
assert d['wave'] == 32 and 'TTFT' in d['reason'], d
print('choose_operating_point: picks the largest SLO-satisfying wave')

In [ ]:
err = {w: 1 - audit_records(all_records, wave=w)['complete'] / max(1, audit_records(all_records, wave=w)['records']) for w in WAVES}
p95d = {w: ttft_percentiles(all_records, w)['p95_ms'] for w in WAVES}
decision = choose_operating_point(tp, p95d, err, ttft_slo_ms=800.0, max_error_rate=0.05)
memo_lines = [
    'TODO: one paragraph — chosen wave vs your Block 0 prediction',
    'TODO: name the resource that bound first and the evidence',
    'TODO: name one alternative explanation you cannot exclude from this run alone',
]
with open(os.path.join(RESULTS, 'decision.json'), 'w') as fh:
    json.dump(dict(decision=decision, memo=memo_lines, profile=PROFILE), fh, indent=1)
print(decision)

## Stretch — A second application arrives (S6)

Application B (the document classifier) now shares the same server. Interference is
measured honestly on A and B separately. You will test one protection: an in-flight
cap K ∈ {1, 2, 4, 8, 16} limiting B's concurrent requests.

Protocol: compare A-only against A+B at each K on a **development** pair, freeze
one K with a written forecast, then run a **distinct held-out** pair once. Accept,
reject and inconclusive are all legitimate outcomes — the freeze is what is graded.

In [ ]:
class InFlightCapRunner:
    """Wrap a runner so at most ``k`` of THIS application's requests are in flight."""
    def __init__(self, runner: object, k: int) -> None:
        """Bind a runner and its in-flight budget.

        Parameters
        ----------
        runner : object
            Any runner with .request(case, max_tokens).
        k : int
            Cap, one of 1/2/4/8/16.

        Returns
        -------
        None
            Wrapper state; see request().
        """
        if k not in (1, 2, 4, 8, 16):
            raise ValueError('K must be one of 1/2/4/8/16')
        self._runner, self._sem = runner, threading.BoundedSemaphore(k)

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Forward one request while holding a cap slot.

        Parameters
        ----------
        case : dict
            Load case for the wrapped runner.
        max_tokens : int
            Output token cap.

        Returns
        -------
        tuple of (str, int)
            Whatever the wrapped runner returns.
        """
        with self._sem:
            return self._runner.request(case, max_tokens)

In [ ]:
def interference_delta(a_alone: Sequence[RequestRecord],
                      a_shared: Sequence[RequestRecord]) -> InterferenceReport:
    """Measure A's degradation when B shares the server.

    Parameters
    ----------
    a_alone, a_shared : sequence of RequestRecord
        A-only journal, and the MERGED A+B journal of the shared wave
        (app_id distinguishes records; the B count is read from it).

    Returns
    -------
    InterferenceReport
        a_throughput_ratio = shared / alone (>1 means A got FASTER — explain or
        suspect the run), a_ttft_p95_ratio = shared / alone, and completed counts.

    Raises
    ------
    ValueError
        If either journal has no completed A record for the wave.
    """
    raise NotImplementedError('TODO: ratios of shared to alone, completed counts')

In [ ]:
alone = [rec('a1', wave=32, arr=0.0, fb=0.2, term=2.0)]
shared = [rec('s1', wave=32, arr=0.0, fb=0.4, term=2.0),
          RequestRecord(request_id='b1', wave=32, client_id=1, app_id='B', case_id='x',
                        input_tokens=480, max_tokens=32, arrival_ts=0.0, first_byte_ts=0.5,
                        terminal_ts=2.0, output_tokens=32, http_status=200, outcome='completed')]
rep = interference_delta(alone, shared)
assert abs(rep['a_ttft_p95_ratio'] - 2.0) < 1e-9 and rep['b_completed_shared'] == 1, rep
print('interference_delta: ratios and B count match the fixture')

In [ ]:
if PROFILE == 'gpu':
    BASE, A6, B6, TMO6 = HttpRunner(), gpu_cases_A, gpu_cases_B, 60.0
else:
    BASE, A6, B6, TMO6 = MockQueueServer(), MOCK_CASES, MOCK_CASES, 12.0
a_alone = run_wave(BASE, 'A', A6, 32, per_client=3, timeout_s=TMO6)
lines: list[str] = []
for k in (1, 2, 4, 8, 16):
    shared = run_shared_wave(BASE, k, A6, B6, n_a=32, n_b=16, per_client=3, timeout_s=TMO6)
    rep = interference_delta(a_alone, shared)
    lines.append(f'K={k:2d}  A throughput ratio={rep["a_throughput_ratio"]:.2f}  '
                 f'A p95 TTFT ratio={rep["a_ttft_p95_ratio"]:.2f}  '
                 f'B completed={rep["b_completed_shared"]}')
print(chr(10).join(lines))

In [ ]:
frozen_k = 4   # TODO: your K from the development table above
k_forecast = 'TODO: predict what the held-out pair will show for A at this K'
with open(os.path.join(RESULTS, 'policy.json'), 'w') as fh:
    json.dump(dict(k=frozen_k, forecast=k_forecast, frozen_at=time.time()), fh, indent=1)
print(f'K={frozen_k} frozen — no policy changes after this point')

In [ ]:
HA = heldout_A if PROFILE == 'gpu' else MOCK_CASES
HB = heldout_B if PROFILE == 'gpu' else MOCK_CASES
heldout = run_shared_wave(BASE, frozen_k, HA, HB, n_a=32, n_b=16, per_client=3, timeout_s=TMO6)
rep = interference_delta(a_alone, heldout)
print(str(rep) + chr(10) + 'verdict — TODO: accept / reject / inconclusive, one sentence tied to the forecast')

In [ ]:
artifacts = {}
for name in ('predictions.json', 'raw-journal.jsonl', 'decision.json',
             'policy.json', 'throughput.png', 'ttft-p95.png', 'outcome-shares.png'):
    p = os.path.join(RESULTS, name)
    if os.path.exists(p):
        artifacts[name] = hashlib.sha256(Path_bytes(p)).hexdigest()
with open(os.path.join(RESULTS, 'manifest.json'), 'w') as fh:
    json.dump(dict(profile=PROFILE, artifacts=artifacts), fh, indent=1)
import shutil  # noqa: E402
shutil.make_archive('/content/t06-' + PROFILE + '-bundle', 'gztar', RESULTS)
print(f'{len(artifacts)} artifacts hashed; profile={PROFILE}; '
      f'bundle: /content/t06-{PROFILE}-bundle.tar.gz')

In [ ]:
# GPU PROFILE ONLY. Owned teardown of the child service started in t06-gpu-server.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
    print('server exit code', server.returncode,
          '- delete the runtime only after the bundle is downloaded')

## What this run does and does not establish

- The **mock** profile validates the pipeline: journals, audits, curves, decisions.
  Its numbers come from a declared synthetic model and are not evidence about any
  real serving system.
- The **gpu** profile (owner-run) produces the measured figures: same driver, same
  analysis, pinned model and frozen public corpus. Only that evidence may enter
  slides, Book or your memo conclusions.
- Free GPU quota on Colab is not guaranteed; the standard-RAM T4 profile is the
  target. Two exact browser rehearsals of this notebook precede any release claim.
- You never edited a raw journal, retried a failed request, or moved a frozen
  decision. Keep it that way in the gpu profile too.